# Eksperimen 2b: Simulasi Pelabelan Otomatis Berbasis Jurnal Biomekanika

Notebook ini mensimulasikan sistem **Automated Ground Truth Labeling** —
setiap tensor pose (64, 33, 3) dievaluasi secara matematis oleh `BiomechanicalValidator`
dan mendapatkan label secara **otomatis** tanpa intervensi manual:
- `is_valid = True`  → **label = 0 (Benar)**
- `is_valid = False` → **label = 1 (Salah)**

**Referensi kriteria biomekanik yang digunakan:**
1. **Chen, K.-Y. et al. (2022)** — *"Fitness Movement Types and Completeness Detection Using a Transfer-Learning-Based Deep Neural Network."*
2. **Rao, P., Asha, C. S., & Rao, R. P. (2023)** — *"Real-time Posture Correction of Squat Exercise: A Deep Learning Approach for Performance Analysis and Error Correction."*
3. **Ko, Y.-M., Nasridinov, A., & Park, S.-H. (2024)** — *"Real-Time AI Posture Correction for Powerlifting Exercises Using YOLOv5 and MediaPipe."*

**Kriteria per gerakan:**

| Gerakan | Kriteria | Threshold | Referensi |
|---|---|---|---|
| **Squat** | Sudut Lutut (Depth) | ≤ 100° | Chen et al. (2022) |
| **Squat** | Knee Valgus (lebar lutut/kaki) | Rasio ≥ 0.85 | Rao et al. (2023) |
| **Bench Press** | Sudut Siku (Elbow ROM) | ≤ 85° | Ko et al. (2024) |
| **Deadlift** | Inklinasi Punggung dari Vertikal | 20° – 60° | Chen et al. (2022), Ko et al. (2024) |

## 1. Import Library & Konfigurasi

In [ ]:
# ============================================================
# Import library dan daftarkan src/ ke sys.path
# ============================================================
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Tambahkan direktori src/ ke Python path
sys.path.insert(0, os.path.abspath("../src"))

from data.biomechanics_validator import BiomechanicalValidator
from data.build_dataset import EXERCISE_VALIDATOR_MAP as _BUILD_MAP

print("Import berhasil.")

# Inisialisasi validator biomekanik
validator = BiomechanicalValidator()

# Peta fungsi validator lokal untuk digunakan di notebook ini
# (sama dengan yang digunakan oleh build_dataset.py)
EXERCISE_VALIDATOR_MAP = {
    "squat"      : validator.validate_squat,
    "deadlift"   : validator.validate_deadlift,
    "benchpress" : validator.validate_benchpress,
    "bench"      : validator.validate_benchpress,
}

print("BiomechanicalValidator berhasil diinisialisasi.")
print(f"  Threshold Squat Depth    : ≤ {validator.SQUAT_DEPTH_THRESHOLD_DEG}°")
print(f"  Threshold Knee Valgus    : Rasio ≥ {validator.SQUAT_VALGUS_RATIO_THRESHOLD}")
print(f"  Threshold Bench Elbow    : ≤ {validator.BENCH_ELBOW_THRESHOLD_DEG}°")
print(f"  Threshold Deadlift Spine : {validator.DEADLIFT_SPINE_MIN_DEG}° – {validator.DEADLIFT_SPINE_MAX_DEG}°")

Import berhasil.
BiomechanicalValidator berhasil diinisialisasi.
  Threshold Squat Depth    : ≤ 100.0°
  Threshold Knee Valgus    : Rasio ≥ 0.85
  Threshold Bench Elbow    : ≤ 85.0°
  Threshold Deadlift Spine : 20.0° – 60.0°


## 2. Simulasi Auto-Labeling pada File .npy Mentah

Cell di bawah mensimulasikan apa yang dilakukan `build_dataset.py` secara internal:
membaca satu atau beberapa file `.npy` hasil preprocessing, lalu memasukkannya ke
`BiomechanicalValidator` untuk mendapatkan label otomatis beserta alasannya.

In [ ]:
# ============================================================
# Definisi path ke folder tensor hasil preprocessing.
# Sesuaikan daftar NPY_FILES dengan file .npy yang benar-benar
# ada di direktori data/processed/tensors/ Anda.
# ============================================================
TENSOR_DIR = Path("../data/processed/tensors")

# Daftar file .npy yang ingin disimulasikan (1-2 sampel cukup untuk demo)
# Format nama file dari build_dataset.py: <NamaLatihan>_<nomor>.npy
# Contoh: Squat_001.npy, Deadlift_001.npy, BenchPress_001.npy
NPY_FILES = sorted(TENSOR_DIR.glob("*.npy"))[:3]  # Ambil maksimum 3 file pertama

if not NPY_FILES:
    display(HTML(
        '<span style="color:orange; font-weight:bold;">'
        '[PERINGATAN] Tidak ada file .npy ditemukan di:<br>'
        f'{TENSOR_DIR.resolve()}<br>'
        'Jalankan build_dataset.py atau notebook 04 terlebih dahulu.'
        '</span>'
    ))
else:
    print(f"Ditemukan {len(list(TENSOR_DIR.glob('*.npy')))} file .npy di {TENSOR_DIR.resolve()}")
    print(f"Akan disimulasikan: {len(NPY_FILES)} file pertama\n")
    print("=" * 72)

    for npy_path in NPY_FILES:
        filename = npy_path.name         # Contoh: Squat_001.npy
        stem     = npy_path.stem.lower() # Contoh: squat_001

        # ── Deteksi jenis gerakan dari nama file ─────────────────────────────
        detected_exercise = None
        validate_fn       = None

        for key, fn in EXERCISE_VALIDATOR_MAP.items():
            if key in stem:
                detected_exercise = key
                validate_fn       = fn
                break

        if validate_fn is None:
            print(f"[SKIP] {filename}: jenis gerakan tidak dikenali dari nama file.\n")
            continue

        # ── Muat tensor dari disk ─────────────────────────────────────────────
        tensor_data = np.load(npy_path)  # Diharapkan shape (64, 33, 3)

        # ── Jalankan validator biomekanik → dapatkan label otomatis ───────────
        is_valid, reason = validate_fn(tensor_data)
        auto_label       = 0 if is_valid else 1
        label_str        = "0 (Benar)" if is_valid else "1 (Salah)"

        # ── Tampilkan hasil dengan format [INFO] sesuai permintaan ────────────
        print(f"[INFO] Video {filename} otomatis diberi label: {label_str}")
        print(f"       Alasan : {reason}")
        print()

    print("=" * 72)
    print("Simulasi selesai.")

Manifest path  : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\data\processed\dataset_manifest.csv
Gerakan yang didukung validator: ['squat', 'deadlift', 'benchpress', 'bench']

[OK] Manifest ditemukan: 4 sampel total
     Label 0 (Benar) : 2 sampel  ← yang akan divalidasi
     Label 1 (Salah) : 2 sampel  ← dilewati


## 3. Verifikasi Konsistensi: Manifest vs. Auto-Label Ulang

Cell ini membaca manifest yang sudah dihasilkan oleh `build_dataset.py`,
lalu **menghitung ulang label** untuk setiap sampel menggunakan validator
dan membandingkannya dengan label yang tersimpan di CSV.
Ketidakcocokan mengindikasikan tensor mungkin rusak atau threshold berubah.

In [ ]:
# ============================================================
# Baca semua sampel dari manifest, hitung ulang label secara
# otomatis, dan bandingkan dengan label yang tersimpan.
# ============================================================
MANIFEST_PATH = Path("../data/processed/dataset_manifest.csv")

if not MANIFEST_PATH.exists():
    display(HTML(
        '<span style="color:orange; font-weight:bold;">'
        '[PERINGATAN] Manifest CSV tidak ditemukan!<br>'
        'Jalankan build_dataset.py atau notebook 04 terlebih dahulu.'
        '</span>'
    ))
else:
    df_manifest = pd.read_csv(MANIFEST_PATH)

    print(f"Memverifikasi {len(df_manifest)} sampel dari manifest...\n")
    print("=" * 72)

    n_match    = 0  # Jumlah label yang cocok
    n_mismatch = 0  # Jumlah label yang tidak cocok (perlu investigasi)
    n_skipped  = 0  # Jumlah sampel yang dilewati (validator/file tidak tersedia)

    mismatch_list = []

    for _, row in df_manifest.iterrows():
        file_path     = row["file_path"]
        stored_label  = int(row["label"])
        filename      = Path(file_path).name
        stem          = Path(file_path).stem.lower()

        # Deteksi jenis gerakan dari nama file
        validate_fn = None
        for key, fn in EXERCISE_VALIDATOR_MAP.items():
            if key in stem:
                validate_fn = fn
                break

        if validate_fn is None or not os.path.exists(file_path):
            n_skipped += 1
            continue

        # Hitung ulang label secara otomatis
        tensor_data      = np.load(file_path)
        is_valid, reason = validate_fn(tensor_data)
        recalc_label     = 0 if is_valid else 1

        if recalc_label == stored_label:
            n_match += 1
        else:
            n_mismatch += 1
            mismatch_list.append({
                "file"          : filename,
                "label_manifest": stored_label,
                "label_recalc"  : recalc_label,
                "reason"        : reason[:80],
            })
            print(f"[INFO] Video {filename} otomatis diberi label: "
                  f"{'0 (Benar)' if recalc_label == 0 else '1 (Salah)'} "
                  f"karena {reason}")

    print("=" * 72)
    print(f"Hasil verifikasi konsistensi label:")
    print(f"  ✓ Cocok    : {n_match}")
    print(f"  ✗ Berbeda  : {n_mismatch}  ← perlu investigasi")
    print(f"  ○ Dilewati : {n_skipped}")

    if n_mismatch > 0:
        display(HTML(
            f'<b style="color:red;">⚠ {n_mismatch} sampel memiliki label berbeda dari manifest!</b><br>'
            'Kemungkinan penyebab: tensor rusak, threshold validator berubah, atau<br>'
            'nama file tidak mengandung kata kunci jenis gerakan yang dikenali.'
        ))
        print("\nDetail ketidakcocokan:")
        print(pd.DataFrame(mismatch_list).to_string(index=False))
    else:
        display(HTML(
            '<b style="color:green;">✓ Semua label di manifest konsisten dengan hasil validator terkini.</b>'
        ))

Memvalidasi 2 sampel berlabel 'Benar'...



Validasi selesai.


## 4. Ringkasan Distribusi Label Otomatis dari Manifest

In [ ]:
# ============================================================
# Tampilkan statistik distribusi label otomatis dari manifest.
# ============================================================
if not MANIFEST_PATH.exists():
    print("Manifest tidak tersedia. Jalankan cell sebelumnya terlebih dahulu.")
else:
    df = pd.read_csv(MANIFEST_PATH)

    total  = len(df)
    benar  = (df["label"] == 0).sum()
    salah  = (df["label"] == 1).sum()

    print("DISTRIBUSI LABEL OTOMATIS (BiomechanicalValidator)")
    print("=" * 50)
    print(f"Total sampel             : {total}")
    print(f"  Label 0 (Benar / PASS) : {benar}  ({benar/total*100:.1f}%)")
    print(f"  Label 1 (Salah / FAIL) : {salah}  ({salah/total*100:.1f}%)")

    if "exercise" in df.columns:
        print()
        print("Per jenis latihan:")
        summary = df.groupby(["exercise", "label"]).size().unstack(fill_value=0)
        summary.columns = [f"Label {c}" for c in summary.columns]
        print(summary.to_string())

    # Plot distribusi sebagai bar chart
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Distribusi Label Otomatis (BiomechanicalValidator)", fontsize=13)

    # Plot 1: Keseluruhan
    axes[0].bar(["Benar (0)", "Salah (1)"], [benar, salah],
                color=["#4CAF50", "#F44336"], edgecolor="black", linewidth=0.8)
    axes[0].set_title("Keseluruhan Dataset")
    axes[0].set_ylabel("Jumlah Sampel")
    for i, v in enumerate([benar, salah]):
        axes[0].text(i, v + 0.1, str(v), ha="center", fontweight="bold")
    axes[0].grid(True, linestyle="--", alpha=0.4, axis="y")

    # Plot 2: Per jenis latihan (jika tersedia)
    if "exercise" in df.columns:
        exercises = df["exercise"].unique()
        x = range(len(exercises))
        benar_per_ex = [int((df[df["exercise"] == ex]["label"] == 0).sum()) for ex in exercises]
        salah_per_ex = [int((df[df["exercise"] == ex]["label"] == 1).sum()) for ex in exercises]
        width = 0.35
        axes[1].bar([i - width/2 for i in x], benar_per_ex, width, label="Benar (0)",
                    color="#4CAF50", edgecolor="black", linewidth=0.8)
        axes[1].bar([i + width/2 for i in x], salah_per_ex, width, label="Salah (1)",
                    color="#F44336", edgecolor="black", linewidth=0.8)
        axes[1].set_xticks(list(x))
        axes[1].set_xticklabels(exercises)
        axes[1].set_title("Per Jenis Latihan")
        axes[1].set_ylabel("Jumlah Sampel")
        axes[1].legend()
        axes[1].grid(True, linestyle="--", alpha=0.4, axis="y")
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.show()

RINGKASAN HASIL VALIDASI BIOMEKANIK
Total sampel 'Benar' yang diperiksa : 2
  ✓ VALID                            : 2
  ✗ GAGAL (perlu dikaji ulang)        : 0
  ○ DILEWATI (tidak diproses)         : 0




Detail per sampel:
                  file exercise  is_valid                                                                                                                         reason
Deadlift_Benar_002.npy deadlift      True Deadlift valid. Inklinasi punggung maksimum = 20.1° (20.0° ≤ x ≤ 60.0°), hip hinge pattern yang aman terdeteksi pada frame 37.
Deadlift_Benar_001.npy deadlift      True Deadlift valid. Inklinasi punggung maksimum = 58.0° (20.0° ≤ x ≤ 60.0°), hip hinge pattern yang aman terdeteksi pada frame 13.


## 5. Verifikasi Unit Test `calculate_angle_3d`

In [ ]:
# ============================================================
# Unit test dasar untuk memverifikasi kebenaran perhitungan
# sudut 3D yang digunakan oleh semua fungsi validator.
# Sudut diketahui secara analitik sehingga dapat dibandingkan.
# ============================================================
print("Verifikasi metode calculate_angle_3d:")
print("-" * 45)

test_cases = [
    # (a, b, c, sudut_diharapkan, deskripsi)
    (np.array([1, 0, 0]), np.array([0, 0, 0]), np.array([0, 1, 0]),  90.0, "Sudut siku-siku (90°)"),
    (np.array([1, 0, 0]), np.array([0, 0, 0]), np.array([1, 0, 0]),   0.0, "Sudut nol (0°) — titik a=c"),
    (np.array([1, 0, 0]), np.array([0, 0, 0]), np.array([-1,0, 0]), 180.0, "Sudut lurus (180°)"),
    (np.array([1, 1, 0]), np.array([0, 0, 0]), np.array([1, 0, 0]),  45.0, "Sudut 45°"),
]

all_passed = True
for a, b, c, expected, desc in test_cases:
    computed = validator.calculate_angle_3d(a, b, c)
    passed   = abs(computed - expected) < 0.01
    status   = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"  [{status}] {desc}: dihitung={computed:.2f}°, diharapkan={expected:.2f}°")

print()
if all_passed:
    display(HTML('<b style="color:green;">[OK] Semua unit test perhitungan sudut lulus.</b>'))
else:
    display(HTML('<b style="color:red;">[FAIL] Ada test yang gagal. Periksa implementasi calculate_angle_3d.</b>'))

Verifikasi metode calculate_angle_3d:
---------------------------------------------
  [PASS] Sudut siku-siku (90°): dihitung=90.00°, diharapkan=90.00°
  [PASS] Sudut nol (0°) — titik a=c: dihitung=0.00°, diharapkan=0.00°
  [PASS] Sudut lurus (180°): dihitung=180.00°, diharapkan=180.00°
  [PASS] Sudut 45°: dihitung=45.00°, diharapkan=45.00°



## 6. Visualisasi: Profil Sudut Lutut pada Sampel Squat Berlabel Otomatis

In [ ]:
# ============================================================
# Plot profil sudut lutut (Hip-Knee-Ankle) sepanjang 64 frame
# untuk sampel Squat apa pun yang ada di manifest.
# Garis threshold ditampilkan untuk memperlihatkan mengapa
# label otomatis ditetapkan sebagai Benar atau Salah.
# ============================================================
if not MANIFEST_PATH.exists():
    print("Manifest tidak tersedia.")
else:
    df_manifest = pd.read_csv(MANIFEST_PATH)

    # Ambil semua sampel Squat (label 0 maupun 1) dari manifest
    squat_samples = df_manifest[
        df_manifest["file_path"].str.lower().str.contains("squat")
    ].reset_index(drop=True)

    if len(squat_samples) == 0:
        print("Tidak ada sampel Squat di manifest.")
        print("Pastikan nama folder mengandung kata 'Squat'.")
    else:
        n_plot = min(len(squat_samples), 4)  # Batasi maksimum 4 subplot
        fig, axes = plt.subplots(1, n_plot, figsize=(7 * n_plot, 5), squeeze=False)
        fig.suptitle(
            "Profil Sudut Lutut — Sampel Squat (Label Otomatis BiomechanicalValidator)",
            fontsize=12
        )

        for i, (_, row) in enumerate(squat_samples.head(n_plot).iterrows()):
            if not os.path.exists(row["file_path"]):
                continue

            tensor        = np.load(row["file_path"])  # (64, 33, 3)
            stored_label  = int(row["label"])
            label_str     = "Benar (0)" if stored_label == 0 else "Salah (1)"
            label_color   = "green" if stored_label == 0 else "red"

            # Hitung sudut Hip-Knee-Ankle per frame (kiri dan kanan)
            angles_left  = validator._get_per_frame_angles(
                tensor,
                BiomechanicalValidator.IDX_L_HIP,
                BiomechanicalValidator.IDX_L_KNEE,
                BiomechanicalValidator.IDX_L_ANKLE,
            )
            angles_right = validator._get_per_frame_angles(
                tensor,
                BiomechanicalValidator.IDX_R_HIP,
                BiomechanicalValidator.IDX_R_KNEE,
                BiomechanicalValidator.IDX_R_ANKLE,
            )
            angles_avg    = (angles_left + angles_right) / 2.0
            deepest_frame = int(np.argmin(angles_avg))

            ax = axes[0][i]
            frames = range(tensor.shape[0])
            ax.plot(frames, angles_left,  "b--", alpha=0.5, label="Lutut Kiri")
            ax.plot(frames, angles_right, "r--", alpha=0.5, label="Lutut Kanan")
            ax.plot(frames, angles_avg,   "k-",  linewidth=2, label="Rata-rata")

            # Tandai posisi terdalam dan threshold
            ax.axvline(x=deepest_frame, color="purple", linestyle=":", linewidth=1.5,
                       label=f"Terdalam (frame {deepest_frame})")
            ax.axhline(y=BiomechanicalValidator.SQUAT_DEPTH_THRESHOLD_DEG,
                       color="orange", linestyle="--", linewidth=1.2,
                       label=f"Threshold ({BiomechanicalValidator.SQUAT_DEPTH_THRESHOLD_DEG}°)")

            ax.set_title(
                f"{Path(row['file_path']).name}\nLabel Otomatis: {label_str}",
                fontsize=9,
                color=label_color,
                fontweight="bold"
            )
            ax.set_xlabel("Frame (0–63)")
            ax.set_ylabel("Sudut Lutut (°)")
            ax.set_ylim(0, 200)
            ax.legend(fontsize=7)
            ax.grid(True, linestyle="--", alpha=0.4)

        plt.tight_layout()
        plt.show()

Tidak ada sampel Squat berlabel Benar di manifest.
Pastikan nama folder latihan mengandung kata 'Squat'.
